# Google Play Apps 2026 (1.1M apps, 1K+ installs): starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/google_play_apps_2026_sample.csv', recursive=True) or ['google_play_apps_2026_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
g = df.groupby('genre').agg(apps=('app_id', 'size'), ads=('contains_ads', 'mean'), iap=('offers_iap', 'mean'), rating=('score', 'median'))
g.sort_values('apps', ascending=False).head(20).round(3)

In [ ]:
order = df.drop_duplicates('installs').assign(n=lambda d: d.installs.str.replace(r'[^0-9]', '', regex=True).astype('int64')).sort_values('n').installs
df.installs.value_counts().reindex(order).plot.bar(figsize=(9, 3.5), logy=True, color='#2563eb', title='Apps by install band (sample)')

In [ ]:
df.assign(year=pd.to_datetime(df.released, errors='coerce').dt.year).groupby('year').size().plot(figsize=(9, 3), title='Apps by release year (sample)')

## Get the full record from the Radar API

The sample keeps the public install band. The API returns the exact install count, year-on-year install growth and the install history. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['app_id'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/apps/profile', headers={'Authorization': f'Bearer {key}'},
                      json={'app_id': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp